# Monitor de alertas: avance, clúster y resultados

Este notebook **solo lee los JSON del flujo**. No abre DuckDB, no recorre Gold, no inicia workers ni recalcula indicadores. Ejecútelo en un **kernel distinto** al notebook 09.

El panel muestra la versión realmente ejecutada. En `alertas-cd-2.0`, D3 mide acumulación en diciembre por entidad y año; un año incompleto no es evaluable. Los parámetros concretos se muestran desde el reporte, sin asumir umbrales fijos.

**Uso:** ejecute todas las celdas para una fotografía. Para seguimiento, cambie `SEGUIR = True` en la configuración y vuelva a ejecutar el panel. Por defecto observa durante 10 minutos; detener el monitor no detiene el ETL.


In [ ]:
from pathlib import Path
import sys, time, math, html
from urllib.parse import urlsplit
from IPython.display import display, HTML

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'secop_pipeline').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Abra este notebook desde la carpeta del proyecto.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from secop_pipeline.alertas_monitor import snapshot, JsonReadCache

SEGUIR = False
INTERVALO_SEGUNDOS = 5
DURACION_MAXIMA_SEGUNDOS = 600
lector = JsonReadCache(max_files=8)
print('Proyecto:', ROOT)


## Panel unificado

Los archivos que no cambian se reutilizan en memoria. Se actualiza una sola salida HTML, y únicamente cuando cambia su contenido. Las estadísticas se muestran en el mismo panel: no hay que esperar a que termine el bucle para verlas.

- El porcentaje y el tiempo restante corresponden **solo al nivel actual de D2**, no al flujo completo.
- Un heartbeat antiguo indica falta de actualización, no confirma un fallo. Si D2 ya terminó, sus cifras se identifican como el último registro.
- `Pendiente` y `No evaluable` no equivalen a cero alertas. Sumar señales puede contar varias veces un contrato.
- Las barras comparan alertas **sobre contratos evaluados**, con conteos y cobertura al lado.


In [ ]:
def esc(value):
    return html.escape(str(value), quote=True)

def numero(value, decimales=0):
    if value is None:
        return '—'
    try:
        return f'{float(value):,.{decimales}f}' if math.isfinite(float(value)) else '—'
    except (TypeError, ValueError):
        return esc(value)

def tabla(cabeceras, filas):
    return ('<table style="border-collapse:collapse;width:100%"><thead><tr>'
            + ''.join('<th style="text-align:left;padding:6px">'+esc(c)+'</th>' for c in cabeceras)
            + '</tr></thead><tbody>'
            + ''.join('<tr>'+''.join('<td style="padding:6px;border-top:1px solid #bbb">'
                      +str(c)+'</td>' for c in fila)+'</tr>' for fila in filas)
            + '</tbody></table>')

def panel(s):
    estado = s.get('estado', 'SIN_EJECUCION')
    version = s.get('version') or 'no registrada'
    partes = ['<section><h2>Alertas · '+esc(estado)+'</h2>',
              '<p>Versión: <b>'+esc(version)+'</b> · Ejecución: '+esc(s.get('flow_run_id', '—'))+'</p>']
    if version != 'alertas-cd-2.0' and estado != 'SIN_EJECUCION':
        partes.append('<p><b>Este reporte no corresponde al modelo 2.0: no interprete su D3 como acumulación de diciembre.</b></p>')
    if 'etapas' not in s:
        return ''.join(partes)+'<p>Todavía no hay una ejecución registrada.</p></section>'
    partes.append('<p>Etapa actual: '+esc(', '.join(s.get('etapa_actual', [])) or 'ninguna')
                  + ' · Etapas completadas: '+numero(s.get('etapas_completadas'))
                  +' (no es porcentaje de tiempo).</p>')
    d = s.get('d2') or {}
    if d:
        activo = estado == 'EN_CURSO' and 'calculate_economic_anomalies' in s.get('etapa_actual', [])
        partes.append('<h3>D2 · '+('Avance actual' if activo else 'Último registro, no actividad actual')+'</h3>')
        pct = d.get('percent')
        if isinstance(pct, (int, float)) and math.isfinite(pct):
            partes.append(f'<progress max="100" value="{min(100,max(0,pct))}" style="width:65%"></progress> '
                          +numero(pct,1)+'% de fechas del nivel '+esc(d.get('level','preparación')))
        partes.append('<p>Bloques: '+numero(d.get('completed_blocks'))+'/'+numero(d.get('total_blocks'))
                      +' · Recuperados de caché: '+numero(d.get('cached_blocks'))+'</p>')
        if activo:
            age = s.get('heartbeat_age_seconds')
            reciente = age is not None and age <= 60
            partes.append('<p>Heartbeat: '+('reciente (≤60 s)' if reciente else 'sin actualización reciente; confirme la actividad del proceso')+'</p>')
            eta = d.get('eta_seconds')
            partes.append('<p>Restante del nivel: '+('~'+numero(eta/60,1)+' minutos' if reciente and eta is not None else 'sin estimación vigente')
                          +'. Niveles posteriores: '+esc(d.get('remaining_levels', []))+'. Luego faltan integración y etapas posteriores.</p>')
        url = s.get('dashboard_url')
        parsed = urlsplit(url) if url else None
        if parsed and parsed.scheme in ('http','https') and parsed.hostname in ('localhost','127.0.0.1','::1'):
            partes.append('<p><a target="_blank" rel="noopener noreferrer" href="'+esc(url)+'">Abrir dashboard Dask</a></p>')
        workers = d.get('workers') or []
        if workers:
            partes.append(tabla(['Worker (último registro)','Estado','CPU %','RAM MiB','Tareas'],
                [[esc(w.get('address','—')),esc(w.get('status','—')),numero(w.get('cpu_percent'),1),
                  numero(w['memory_bytes']/2**20,1) if w.get('memory_bytes') is not None else '—',
                  esc(w.get('tasks','—'))] for w in workers]))
    else:
        partes.append('<p>Aún no hay avance por bloques: preparación o ejecución con código anterior.</p>')

    resumen = s.get('resumen') or {}
    dimensiones = resumen.get('dimensiones') or s.get('hallazgos') or {}
    nombres = {'D1':'Concentración', 'D2':'Valor atípico',
               'D3':'Acumulación de diciembre' if version == 'alertas-cd-2.0' else 'Definición de la versión registrada'}
    filas = []
    for k in ('D1','D2','D3'):
        v = dimensiones.get(k)
        if v is None:
            filas.append([k+' · '+nombres[k], 'Pendiente', '—','—','—','—'])
            continue
        tasa = v.get('alertas_sobre_evaluados_pct')
        barra = 'No calculable'
        if isinstance(tasa,(int,float)) and math.isfinite(tasa):
            barra = (f'<progress max="100" value="{min(100,max(0,tasa))}" style="width:85px"></progress> '
                     +numero(tasa,2)+'%')
        filas.append([k+' · '+nombres[k],numero(v.get('alertas')),numero(v.get('evaluados')),
                      numero(v.get('no_evaluables')),numero(v.get('cobertura_pct'),2)+'%',barra])
    partes.append('<h3>Hallazgos por indicador · contratos</h3>'+tabla(
        ['Indicador','Alertas','Evaluados','No evaluables','Cobertura','Alertas / evaluados'],filas))
    if resumen:
        partes.append('<h3>Resultados conjuntos</h3>'+tabla(['Medida','Contratos / señales'],[
            ['Universo',numero(resumen.get('contratos_resultado'))],
            ['Contratos únicos con alguna señal',numero(resumen.get('contratos_con_alguna_senal'))],
            ['Señales sumadas (un contrato puede aportar varias)',numero(resumen.get('total_senales'))],
            ['Evaluados en los tres indicadores',numero(resumen.get('contratos_evaluados_completos'))]]))
        for titulo, clave in [('Clasificación final','clasificaciones'),('Coincidencias (pares incluyen el triple)','coincidencias'),
                              ('D3: entidades-año, no contratos','d3_entidades_anio')]:
            valores = resumen.get(clave) or {}
            if valores:
                partes.append('<h4>'+titulo+'</h4>'+tabla(['Categoría','Cantidad'],[[esc(k),numero(v)] for k,v in valores.items()]))
        combinaciones = resumen.get('combinaciones') or []
        if combinaciones:
            def senal(v):
                return 'No evaluable' if v is None else 'Alerta' if v == 1 else 'Sin alerta'
            partes.append('<details><summary>Combinaciones excluyentes de señales</summary>'+tabla(
                ['D1','D2','D3','Clasificación','Contratos','% universo'],
                [[senal(r.get(k)) for k in ('D1','D2','D3')]+[esc(r.get('clasificacion','—')),
                  numero(r.get('contratos')),numero(r.get('porcentaje_total'),2)] for r in combinaciones])+'</details>')
    else:
        partes.append('<p>La clasificación y coincidencias estarán disponibles al terminar el cálculo.</p>')
    parametros = s.get('parametros') or {}
    partes.append('<details><summary>Etapas y parámetros D3 de esta ejecución</summary>'+tabla(
        ['Etapa','Estado'],[[esc(k),esc(v.get('estado','—'))] for k,v in s.get('etapas',{}).items()])
        +tabla(['Parámetro','Valor'],[[esc(k),esc(v)] for k,v in parametros.items() if k.startswith('d3_')])+'</details>')
    partes.append('<p>Las señales orientan la revisión; no demuestran corrupción. Los datos pendientes y no evaluables no representan ausencia de riesgo.</p></section>')
    return ''.join(partes)


In [ ]:
# Reejecute esta celda para una nueva fotografía o sesión de seguimiento.
if INTERVALO_SEGUNDOS < 2 or DURACION_MAXIMA_SEGUNDOS <= 0:
    raise ValueError('Use intervalo >= 2 segundos y duración máxima positiva.')
salida = display(HTML('<p>Leyendo estado…</p>'), display_id=True)
anterior = None
limite = time.monotonic() + DURACION_MAXIMA_SEGUNDOS
try:
    while True:
        estado = None
        try:
            s = snapshot(ROOT, reader=lector)
            contenido = panel(s)
            estado = s.get('estado')
        except (OSError, ValueError, KeyError, TypeError) as exc:
            contenido = '<p>Reporte no disponible o en actualización. Reintente. Detalle: '+esc(exc)+'</p>'
        if contenido != anterior:
            salida.update(HTML(contenido))
            anterior = contenido
        if not SEGUIR or estado in ('COMPLETADA','FALLIDA','CANCELADA'):
            break
        restante = limite - time.monotonic()
        if restante <= 0:
            print('Fin del tiempo de observación. Reejecute esta celda para continuar; el ETL sigue independiente.')
            break
        time.sleep(min(INTERVALO_SEGUNDOS, restante))
except KeyboardInterrupt:
    print('Monitor detenido; no se ha detenido el flujo de alertas.')


## Interpretación y continuidad

**D3 versión 2.0:** compara la participación de diciembre de una entidad-año con su historia de años completos. La alerta se asigna a los contratos de diciembre que cumplen la regla. La tabla de entidades-año evita confundir diez contratos señalados de una misma entidad-año con diez casos independientes. Los límites se consultan en el desplegable de parámetros.

**Si el estado parece detenido:** un reporte `EN_CURSO` puede sobrevivir al cierre del kernel. Revise el proceso y el notebook 09; este monitor no consulta procesos ni confirma su actividad. Una desconexión del dashboard tampoco demuestra que el ETL falló.

**Para cargar cambios de código:** detenga la ejecución anterior del notebook 09, espere su cierre y reinicie su kernel antes de relanzar. Reinicie también el kernel de este monitor si ya había importado su módulo. No borre `salida/alertas`: allí están los checkpoints. No elimine un lock de una ejecución activa.

Alternativa para seguimiento desde otra terminal en la raíz del proyecto:
```powershell
python scripts/monitor_alertas.py --watch --open-dashboard
```

Este notebook no exporta resultados. La caché del monitor dura únicamente mientras vive su kernel y es distinta de los checkpoints del ETL.
